# 09 — Windowed Reconstruction Stability

**Repo:** `github.com/thinkthoughts/prime-numbers-lab`  
**Purpose:** test reconstruction stability across sliding intervals, so global density does not hide local failures.

Notebook chain:

```text
07: global density-guided reconstruction
08: local gap-guided reconstruction
09: windowed reconstruction stability
```

Notebook 08 showed:

> local-gap scoring improves gap diagnostics only marginally when global metrics are saturated.

Notebook 09 asks:

> Do density-only and local-gap reconstruction remain stable inside local windows?

Core claim:

> Global reconstruction can look successful while local windows expose density, gap, and stability failures.

## 0. Setup

Artifact structure:

```text
09_windowed_reconstruction_stability/
├── data/
├── docs/
├── figures/
└── tex/
```

Root export:

```text
09_windowed_reconstruction_stability_export.zip
```

In [ ]:
from pathlib import Path
import json
import math
import zipfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

NOTEBOOK_ID = "09_windowed_reconstruction_stability"
NOTEBOOK_NUM = NOTEBOOK_ID.split("_")[0]
NOTEBOOK_TITLE = "Windowed Reconstruction Stability"

OUT = Path(NOTEBOOK_ID)
DATA_DIR = OUT / "data"
DOCS_DIR = OUT / "docs"
FIG_DIR = OUT / "figures"
TEX_DIR = OUT / "tex"

for d in [DATA_DIR, DOCS_DIR, FIG_DIR, TEX_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Artifact directory: {OUT.resolve()}")

## 1. Premise

Notebook 07 and 08 used global metrics:

\[
\pi(N), \quad F_1, \quad \text{global density drift}, \quad \text{global gap drift}.
\]

Those metrics can saturate.

Notebook 09 switches to local windows:

\[
[x, x+\Delta]
\]

and measures:

\[
\pi_{\hat{P}}(x+\Delta)-\pi_{\hat{P}}(x)
\]

against true local prime count.

This tests whether reconstruction is locally stable, not merely globally correct.

## 2. Window metrics

For window \(W=[a,b]\):

\[
density\_error(W)=
\frac{|\ |\hat{P}\cap W|-|P\cap W|\ |}{|P\cap W|}
\]

Local gap drift:

\[
gap\_drift(W)=
\operatorname{mean}_{p_i\in W}
\frac{|g_i-\log p_i|}{\log p_i}
\]

Window precision:

\[
precision(W)=
\frac{|\hat{P}\cap P\cap W|}{|\hat{P}\cap W|}
\]

Window recovery:

\[
recovery(W)=
\frac{|\hat{P}\cap P\cap W|}{|P\cap W|}
\]

In [ ]:
# Parameters

N_MAX = 200_000
RANDOM_SEED = 9423

SCENARIOS = [
    {"name": "mixed_keep_75_noise_25", "kind": "mixed", "keep_fraction": 0.75, "noise_fraction": 0.25},
    {"name": "mixed_keep_50_noise_50", "kind": "mixed", "keep_fraction": 0.50, "noise_fraction": 0.50},
    {"name": "mixed_keep_25_noise_100", "kind": "mixed", "keep_fraction": 0.25, "noise_fraction": 1.00},
    {"name": "biased_low_missing_high_noise", "kind": "biased_range", "keep_fraction": 0.50, "noise_fraction": 0.50},
    {"name": "adversarial_mod6_noise_50", "kind": "adversarial_mod6", "keep_fraction": 0.50, "noise_fraction": 0.50},
    {"name": "adversarial_mod6_noise_100", "kind": "adversarial_mod6", "keep_fraction": 0.25, "noise_fraction": 1.00},
]

Q_FILTER = int(math.sqrt(N_MAX))
COMPLETION_BIN_COUNT = 32

# Local-gap reconstruction weights from Notebook 08.
W_DENSITY = 0.45
W_GAP = 0.40
W_NEIGHBOR = 0.15

# Sliding windows.
WINDOW_WIDTH = 10_000
WINDOW_STEP = 5_000
WINDOW_EDGES = [(a, min(a + WINDOW_WIDTH, N_MAX)) for a in range(2, N_MAX, WINDOW_STEP) if a + 100 <= N_MAX]

rng = np.random.default_rng(RANDOM_SEED)

params = {
    "N_MAX": N_MAX,
    "RANDOM_SEED": RANDOM_SEED,
    "SCENARIOS": SCENARIOS,
    "Q_FILTER": Q_FILTER,
    "COMPLETION_BIN_COUNT": COMPLETION_BIN_COUNT,
    "W_DENSITY": W_DENSITY,
    "W_GAP": W_GAP,
    "W_NEIGHBOR": W_NEIGHBOR,
    "WINDOW_WIDTH": WINDOW_WIDTH,
    "WINDOW_STEP": WINDOW_STEP,
    "WINDOW_COUNT": len(WINDOW_EDGES),
    "NOTEBOOK_ID": NOTEBOOK_ID,
    "NOTEBOOK_TITLE": NOTEBOOK_TITLE,
}

params

## 3. Reference primes and stress-test observations

Same stress-test family as Notebook 08:

- mixed corruption
- biased range corruption
- adversarial mod-6-passing composite noise

In [ ]:
def simple_sieve(n: int) -> np.ndarray:
    if n < 2:
        return np.array([], dtype=int)
    s = np.ones(n + 1, dtype=bool)
    s[:2] = False
    for i in range(2, int(math.sqrt(n)) + 1):
        if s[i]:
            s[i*i:n+1:i] = False
    return np.nonzero(s)[0]

reference_primes = simple_sieve(N_MAX)
prime_set = set(reference_primes.tolist())
universe = np.arange(2, N_MAX + 1)
composites = np.array([n for n in universe if n not in prime_set], dtype=int)
mod6_composites = composites[np.isin(composites % 6, [1, 5])]

def make_observation(cfg: dict) -> dict:
    name = cfg["name"]
    kind = cfg["kind"]
    keep_fraction = cfg["keep_fraction"]
    noise_fraction = cfg["noise_fraction"]

    keep_count = int(round(keep_fraction * len(reference_primes)))
    noise_count = int(round(noise_fraction * len(reference_primes)))

    if kind == "biased_range":
        midpoint = N_MAX // 2
        low_primes = reference_primes[reference_primes <= midpoint]
        high_primes = reference_primes[reference_primes > midpoint]

        low_keep_count = min(len(low_primes), int(round(0.90 * len(low_primes))))
        remaining_keep = max(0, keep_count - low_keep_count)
        high_keep_count = min(len(high_primes), remaining_keep)

        kept_low = rng.choice(low_primes, size=low_keep_count, replace=False)
        kept_high = rng.choice(high_primes, size=high_keep_count, replace=False) if high_keep_count else np.array([], dtype=int)

        high_composites = composites[composites > midpoint]
        noise_values = rng.choice(high_composites, size=min(noise_count, len(high_composites)), replace=False)

        values = np.sort(np.unique(np.concatenate([kept_low, kept_high, noise_values])))

    elif kind == "adversarial_mod6":
        kept_primes = rng.choice(reference_primes, size=keep_count, replace=False)
        noise_values = rng.choice(mod6_composites, size=min(noise_count, len(mod6_composites)), replace=False)
        values = np.sort(np.unique(np.concatenate([kept_primes, noise_values])))

    else:
        kept_primes = rng.choice(reference_primes, size=keep_count, replace=False)
        noise_values = rng.choice(composites, size=min(noise_count, len(composites)), replace=False)
        values = np.sort(np.unique(np.concatenate([kept_primes, noise_values])))

    return {
        "name": name,
        "kind": kind,
        "keep_fraction": keep_fraction,
        "noise_fraction": noise_fraction,
        "values": values,
    }

observations = [make_observation(cfg) for cfg in SCENARIOS]

summary = {
    "n_max": int(N_MAX),
    "prime_count": int(len(reference_primes)),
    "composite_count": int(len(composites)),
    "mod6_composite_count": int(len(mod6_composites)),
    "scenario_count": int(len(observations)),
    "window_count": int(len(WINDOW_EDGES)),
}

summary, [(o["name"], len(o["values"])) for o in observations]

## 4. Filters and reconstruction functions

In [ ]:
def residue_filter(values: np.ndarray) -> np.ndarray:
    values = np.asarray(values, dtype=int)
    keep = (values == 2) | (values == 3) | np.isin(values % 6, [1, 5])
    return np.sort(values[keep])

def passes_sieve(values: np.ndarray, q_max: int) -> np.ndarray:
    values = np.asarray(values, dtype=int)
    keep = np.ones(len(values), dtype=bool)
    filter_primes = reference_primes[reference_primes <= q_max]
    for q in filter_primes:
        keep &= ((values == q) | (values % q != 0))
    return keep

def sieve_filter(values: np.ndarray, q_max: int) -> np.ndarray:
    return np.sort(values[passes_sieve(values, q_max)])

def pi_model(x: np.ndarray | float) -> np.ndarray:
    x = np.asarray(x, dtype=float)
    safe = np.maximum(x, 3.0)
    denom = np.log(safe) - 1.0
    denom = np.maximum(denom, 1.0)
    return safe / denom

def candidate_pool(q_max: int) -> np.ndarray:
    candidates = residue_filter(universe)
    return sieve_filter(candidates, q_max=q_max)

POOL = candidate_pool(Q_FILTER)

def nearest_gap_scores(candidates: np.ndarray, current_values: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    candidates = np.asarray(candidates, dtype=int)
    current_values = np.sort(np.unique(current_values.astype(int)))
    if len(current_values) == 0:
        return np.ones(len(candidates)), np.ones(len(candidates))

    pos = np.searchsorted(current_values, candidates)
    left_neighbor = np.where(pos > 0, current_values[np.maximum(pos - 1, 0)], -10**12)
    right_neighbor = np.where(pos < len(current_values), current_values[np.minimum(pos, len(current_values)-1)], 10**12)

    left_gap = candidates - left_neighbor
    right_gap = right_neighbor - candidates
    expected = np.maximum(np.log(np.maximum(candidates, 3)), 1.0)

    finite_left = left_neighbor > 0
    finite_right = right_neighbor < 10**11

    left_score = np.where(finite_left, np.exp(-np.abs(left_gap - expected) / expected), 0.5)
    right_score = np.where(finite_right, np.exp(-np.abs(right_gap - expected) / expected), 0.5)
    gap_score = 0.5 * (left_score + right_score)

    nearest = np.minimum(np.where(finite_left, left_gap, expected), np.where(finite_right, right_gap, expected))
    neighbor_score = 1.0 - np.exp(-nearest / expected)

    return gap_score, neighbor_score

def density_only_completion(filtered_values: np.ndarray, target_count: int) -> tuple[np.ndarray, pd.DataFrame]:
    filtered_values = np.sort(np.unique(filtered_values.astype(int)))
    existing = set(filtered_values.tolist())
    available = np.array([v for v in POOL if v not in existing], dtype=int)

    needed_total = max(0, int(target_count - len(filtered_values)))
    if needed_total == 0 or len(available) == 0:
        return filtered_values, pd.DataFrame(columns=["candidate", "method", "bin_left", "bin_right", "score"])

    edges = np.unique(np.linspace(2, N_MAX, COMPLETION_BIN_COUNT + 1).astype(int))
    rows = []

    for left, right in zip(edges[:-1], edges[1:]):
        bin_available = available[(available >= left) & (available <= right)]
        if len(bin_available) == 0:
            continue

        observed_count = int(np.sum((filtered_values >= left) & (filtered_values <= right)))
        model_bin_count = int(max(0, round(pi_model(right) - pi_model(left))))
        bin_need = max(0, model_bin_count - observed_count)
        if bin_need == 0:
            continue

        center = 0.5 * (left + right)
        scale = max(1.0, right - left)
        scores = 1.0 - np.abs(bin_available - center) / scale
        scores = scores + rng.normal(0, 1e-6, size=len(scores))
        order = np.argsort(scores)[::-1]
        chosen = bin_available[order[:min(bin_need, len(bin_available))]]
        score_lookup = dict(zip(bin_available.tolist(), scores.tolist()))

        for c in chosen:
            rows.append({
                "candidate": int(c),
                "method": "density_only",
                "bin_left": int(left),
                "bin_right": int(right),
                "score": float(score_lookup[int(c)]),
            })

    df = pd.DataFrame(rows)
    if len(df) > 0:
        df = df.sort_values("score", ascending=False).head(needed_total)
        selected = df["candidate"].to_numpy(dtype=int)
    else:
        selected = np.array([], dtype=int)

    reconstructed = np.sort(np.unique(np.concatenate([filtered_values, selected])))
    return reconstructed, df

def local_gap_completion(filtered_values: np.ndarray, target_count: int) -> tuple[np.ndarray, pd.DataFrame]:
    current = np.sort(np.unique(filtered_values.astype(int)))
    existing = set(current.tolist())
    available = np.array([v for v in POOL if v not in existing], dtype=int)

    needed_total = max(0, int(target_count - len(current)))
    if needed_total == 0 or len(available) == 0:
        return current, pd.DataFrame(columns=["candidate", "method", "bin_left", "bin_right", "score"])

    edges = np.unique(np.linspace(2, N_MAX, COMPLETION_BIN_COUNT + 1).astype(int))
    rows = []

    for left, right in zip(edges[:-1], edges[1:]):
        bin_available = available[(available >= left) & (available <= right)]
        if len(bin_available) == 0:
            continue

        observed_count = int(np.sum((current >= left) & (current <= right)))
        model_bin_count = int(max(0, round(pi_model(right) - pi_model(left))))
        bin_need = max(0, model_bin_count - observed_count)
        if bin_need == 0:
            continue

        center = 0.5 * (left + right)
        scale = max(1.0, right - left)
        density_score = 1.0 - np.abs(bin_available - center) / scale
        gap_score, neighbor_score = nearest_gap_scores(bin_available, current)

        score = (
            W_DENSITY * density_score
            + W_GAP * gap_score
            + W_NEIGHBOR * neighbor_score
            + rng.normal(0, 1e-6, size=len(bin_available))
        )

        order = np.argsort(score)[::-1]
        chosen = bin_available[order[:min(bin_need, len(bin_available))]]
        lookup = {
            int(v): (float(score[i]), float(density_score[i]), float(gap_score[i]), float(neighbor_score[i]))
            for i, v in enumerate(bin_available)
        }

        for c in chosen:
            s, ds, gs, ns = lookup[int(c)]
            rows.append({
                "candidate": int(c),
                "method": "local_gap",
                "bin_left": int(left),
                "bin_right": int(right),
                "score": s,
                "density_score": ds,
                "gap_score": gs,
                "neighbor_score": ns,
            })

    df = pd.DataFrame(rows)
    if len(df) > 0:
        df = df.sort_values("score", ascending=False).head(needed_total)
        selected = df["candidate"].to_numpy(dtype=int)
    else:
        selected = np.array([], dtype=int)

    reconstructed = np.sort(np.unique(np.concatenate([current, selected])))
    return reconstructed, df

print("Reconstruction functions ready")

## 5. Build reconstructed sets

Methods:

1. raw
2. sieve-cleaned
3. density-only reconstruction
4. local-gap reconstruction

In [ ]:
target_count = len(reference_primes)

reconstructed_sets = {}
candidate_completion_frames = []

for obs in observations:
    scenario = obs["name"]
    kind = obs["kind"]

    raw = np.sort(obs["values"])
    cleaned = sieve_filter(residue_filter(raw), q_max=Q_FILTER)

    density_recon, density_completions = density_only_completion(cleaned, target_count)
    local_recon, local_completions = local_gap_completion(cleaned, target_count)

    reconstructed_sets[(scenario, "raw")] = raw
    reconstructed_sets[(scenario, "sieve_cleaned")] = cleaned
    reconstructed_sets[(scenario, "density_only")] = density_recon
    reconstructed_sets[(scenario, "local_gap")] = local_recon

    density_completions["scenario"] = scenario
    density_completions["kind"] = kind
    local_completions["scenario"] = scenario
    local_completions["kind"] = kind

    candidate_completion_frames.extend([density_completions, local_completions])

candidate_completions_df = pd.concat(candidate_completion_frames, ignore_index=True) if candidate_completion_frames else pd.DataFrame()

[(k, len(v)) for k, v in list(reconstructed_sets.items())[:8]]

## 6. Windowed metrics

Compute local density, gap, precision, recovery, and F1 in sliding windows.

In [ ]:
def values_in_window(values: np.ndarray, a: int, b: int) -> np.ndarray:
    values = np.sort(values)
    i = np.searchsorted(values, a, side="left")
    j = np.searchsorted(values, b, side="right")
    return values[i:j]

def window_gap_drift(values_window: np.ndarray) -> float:
    vals = np.sort(np.unique(values_window.astype(int)))
    if len(vals) < 3:
        return float("nan")
    gaps = np.diff(vals)
    anchors = vals[:-1]
    expected = np.maximum(np.log(np.maximum(anchors, 3)), 1.0)
    rel = np.abs(gaps - expected) / expected
    return float(np.mean(np.clip(rel, 0, 20)))

def window_metrics(values: np.ndarray, scenario: str, kind: str, method: str, a: int, b: int) -> dict:
    v_win = values_in_window(values, a, b)
    p_win = values_in_window(reference_primes, a, b)

    v_set = set(v_win.tolist())
    p_set = set(p_win.tolist())

    tp = len(v_set & p_set)
    fp = len(v_set - p_set)
    fn = len(p_set - v_set)

    true_count = len(p_win)
    obs_count = len(v_win)

    density_error = abs(obs_count - true_count) / true_count if true_count else float("nan")
    precision = tp / obs_count if obs_count else float("nan")
    recovery = tp / true_count if true_count else float("nan")
    f1 = 2 * precision * recovery / (precision + recovery) if (precision + recovery) else float("nan")
    gap_drift = window_gap_drift(v_win)

    return {
        "scenario": scenario,
        "kind": kind,
        "method": method,
        "window_left": int(a),
        "window_right": int(b),
        "window_mid": float(0.5 * (a + b)),
        "true_prime_count": int(true_count),
        "observed_count": int(obs_count),
        "true_positive": int(tp),
        "false_positive": int(fp),
        "false_negative": int(fn),
        "density_error": float(density_error),
        "precision": float(precision),
        "recovery": float(recovery),
        "f1": float(f1),
        "gap_drift": float(gap_drift),
    }

window_rows = []
method_order = ["raw", "sieve_cleaned", "density_only", "local_gap"]

for obs in observations:
    scenario = obs["name"]
    kind = obs["kind"]

    for method in method_order:
        values = reconstructed_sets[(scenario, method)]
        for a, b in WINDOW_EDGES:
            window_rows.append(window_metrics(values, scenario, kind, method, a, b))

window_metrics_df = pd.DataFrame(window_rows)

window_metrics_df.head()

## 7. Stability summaries

Summarize worst-case and mean local behavior.

In [ ]:
summary_rows = []

for (scenario, kind, method), sub in window_metrics_df.groupby(["scenario", "kind", "method"]):
    summary_rows.append({
        "scenario": scenario,
        "kind": kind,
        "method": method,
        "mean_density_error": float(sub["density_error"].mean()),
        "max_density_error": float(sub["density_error"].max()),
        "mean_gap_drift": float(sub["gap_drift"].mean(skipna=True)),
        "max_gap_drift": float(sub["gap_drift"].max(skipna=True)),
        "mean_precision": float(sub["precision"].mean(skipna=True)),
        "mean_recovery": float(sub["recovery"].mean(skipna=True)),
        "mean_f1": float(sub["f1"].mean(skipna=True)),
        "unstable_density_windows_10pct": int((sub["density_error"] > 0.10).sum()),
        "unstable_gap_windows_gt1": int((sub["gap_drift"] > 1.0).sum()),
    })

window_summary_df = pd.DataFrame(summary_rows)

delta_rows = []
for scenario in [cfg["name"] for cfg in SCENARIOS]:
    density = window_summary_df[(window_summary_df["scenario"] == scenario) & (window_summary_df["method"] == "density_only")].iloc[0]
    local = window_summary_df[(window_summary_df["scenario"] == scenario) & (window_summary_df["method"] == "local_gap")].iloc[0]
    delta_rows.append({
        "scenario": scenario,
        "kind": local["kind"],
        "delta_mean_density_error_local_minus_density": float(local["mean_density_error"] - density["mean_density_error"]),
        "delta_max_density_error_local_minus_density": float(local["max_density_error"] - density["max_density_error"]),
        "delta_mean_gap_drift_local_minus_density": float(local["mean_gap_drift"] - density["mean_gap_drift"]),
        "delta_max_gap_drift_local_minus_density": float(local["max_gap_drift"] - density["max_gap_drift"]),
        "delta_unstable_density_windows_local_minus_density": int(local["unstable_density_windows_10pct"] - density["unstable_density_windows_10pct"]),
        "delta_unstable_gap_windows_local_minus_density": int(local["unstable_gap_windows_gt1"] - density["unstable_gap_windows_gt1"]),
    })

window_deltas_df = pd.DataFrame(delta_rows)

measurement = {
    "mean_window_density_error_density_only": float(window_summary_df[window_summary_df["method"] == "density_only"]["mean_density_error"].mean()),
    "mean_window_density_error_local_gap": float(window_summary_df[window_summary_df["method"] == "local_gap"]["mean_density_error"].mean()),
    "mean_window_gap_drift_density_only": float(window_summary_df[window_summary_df["method"] == "density_only"]["mean_gap_drift"].mean()),
    "mean_window_gap_drift_local_gap": float(window_summary_df[window_summary_df["method"] == "local_gap"]["mean_gap_drift"].mean()),
    "mean_delta_gap_drift_local_minus_density": float(window_deltas_df["delta_mean_gap_drift_local_minus_density"].mean()),
    "mean_delta_density_error_local_minus_density": float(window_deltas_df["delta_mean_density_error_local_minus_density"].mean()),
}

cgcs = {
    "score": 1.0 / (
        1.0
        + measurement["mean_window_density_error_local_gap"]
        + measurement["mean_window_gap_drift_local_gap"]
    ),
    "definition": "windowed stability score = 1/(1 + mean density error + mean gap drift) for local-gap reconstruction",
    "interpretation": "Higher score means better local density and gap stability.",
}

measurement, cgcs, window_summary_df.head(), window_deltas_df

## 8. Figure 1 — mean window density error by method

Lower is better.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

for scenario in [cfg["name"] for cfg in SCENARIOS]:
    sub = window_summary_df[window_summary_df["scenario"] == scenario].set_index("method").loc[method_order]
    ax.plot(method_order, sub["mean_density_error"], marker="o", label=scenario)

ax.set_title("Mean window density error by method")
ax.set_xlabel("method")
ax.set_ylabel("mean local density error")
ax.tick_params(axis="x", rotation=20)
ax.legend(fontsize=7)
ax.grid(True, alpha=0.3)

fig1_path = FIG_DIR / f"{NOTEBOOK_NUM}_mean_window_density_error_by_method.png"
fig.savefig(fig1_path, dpi=180, bbox_inches="tight")
plt.show()

fig1_path

## 9. Figure 2 — mean window gap drift by method

Lower is better.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

for scenario in [cfg["name"] for cfg in SCENARIOS]:
    sub = window_summary_df[window_summary_df["scenario"] == scenario].set_index("method").loc[method_order]
    ax.plot(method_order, sub["mean_gap_drift"], marker="o", label=scenario)

ax.set_title("Mean window gap drift by method")
ax.set_xlabel("method")
ax.set_ylabel("mean local gap drift")
ax.tick_params(axis="x", rotation=20)
ax.legend(fontsize=7)
ax.grid(True, alpha=0.3)

fig2_path = FIG_DIR / f"{NOTEBOOK_NUM}_mean_window_gap_drift_by_method.png"
fig.savefig(fig2_path, dpi=180, bbox_inches="tight")
plt.show()

fig2_path

## 10. Figure 3 — local density error profile

Representative hardest scenario:

```text
adversarial_mod6_noise_100
```

In [ ]:
representative = "adversarial_mod6_noise_100"
fig, ax = plt.subplots(figsize=(12, 6))

for method in method_order:
    sub = window_metrics_df[
        (window_metrics_df["scenario"] == representative)
        & (window_metrics_df["method"] == method)
    ].sort_values("window_mid")
    ax.plot(sub["window_mid"], sub["density_error"], marker="o", markersize=3, label=method)

ax.set_title(f"Window density error profile — {representative}")
ax.set_xlabel("window midpoint")
ax.set_ylabel("density error")
ax.legend()
ax.grid(True, alpha=0.3)

fig3_path = FIG_DIR / f"{NOTEBOOK_NUM}_window_density_error_profile.png"
fig.savefig(fig3_path, dpi=180, bbox_inches="tight")
plt.show()

fig3_path

## 11. Figure 4 — local gap drift profile

Same representative scenario.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

for method in method_order:
    sub = window_metrics_df[
        (window_metrics_df["scenario"] == representative)
        & (window_metrics_df["method"] == method)
    ].sort_values("window_mid")
    ax.plot(sub["window_mid"], sub["gap_drift"], marker="o", markersize=3, label=method)

ax.set_title(f"Window gap drift profile — {representative}")
ax.set_xlabel("window midpoint")
ax.set_ylabel("gap drift")
ax.legend()
ax.grid(True, alpha=0.3)

fig4_path = FIG_DIR / f"{NOTEBOOK_NUM}_window_gap_drift_profile.png"
fig.savefig(fig4_path, dpi=180, bbox_inches="tight")
plt.show()

fig4_path

## 12. Figure 5 — density-only vs local-gap window delta

Negative means local-gap improved local gap drift.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
labels = window_deltas_df["scenario"].str.replace("_", "\n")

ax.bar(labels, window_deltas_df["delta_mean_gap_drift_local_minus_density"])
ax.axhline(0, linestyle="--", linewidth=1)
ax.set_title("Local-gap minus density-only mean window gap drift")
ax.set_xlabel("scenario")
ax.set_ylabel("delta mean gap drift")
ax.tick_params(axis="x", rotation=45)
ax.grid(True, axis="y", alpha=0.3)

fig5_path = FIG_DIR / f"{NOTEBOOK_NUM}_window_gap_drift_delta.png"
fig.savefig(fig5_path, dpi=180, bbox_inches="tight")
plt.show()

fig5_path

## 13. Figure 6 — unstable density windows

Count windows where:

\[
density\_error > 0.10
\]

In [ ]:
unstable_density = window_summary_df.pivot(index="scenario", columns="method", values="unstable_density_windows_10pct").loc[[cfg["name"] for cfg in SCENARIOS], method_order]

fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(len(unstable_density.index))
width = 0.2

for i, method in enumerate(method_order):
    ax.bar(x + (i - 1.5) * width, unstable_density[method], width, label=method)

ax.set_xticks(x)
ax.set_xticklabels([s.replace("_", "\n") for s in unstable_density.index], rotation=45, ha="right")
ax.set_title("Unstable density windows")
ax.set_xlabel("scenario")
ax.set_ylabel("count of windows with density error > 0.10")
ax.legend()
ax.grid(True, axis="y", alpha=0.3)

fig6_path = FIG_DIR / f"{NOTEBOOK_NUM}_unstable_density_windows.png"
fig.savefig(fig6_path, dpi=180, bbox_inches="tight")
plt.show()

fig6_path

## 14. Figure 7 — unstable gap windows

Count windows where:

\[
gap\_drift > 1
\]

In [ ]:
unstable_gap = window_summary_df.pivot(index="scenario", columns="method", values="unstable_gap_windows_gt1").loc[[cfg["name"] for cfg in SCENARIOS], method_order]

fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(len(unstable_gap.index))
width = 0.2

for i, method in enumerate(method_order):
    ax.bar(x + (i - 1.5) * width, unstable_gap[method], width, label=method)

ax.set_xticks(x)
ax.set_xticklabels([s.replace("_", "\n") for s in unstable_gap.index], rotation=45, ha="right")
ax.set_title("Unstable gap windows")
ax.set_xlabel("scenario")
ax.set_ylabel("count of windows with gap drift > 1")
ax.legend()
ax.grid(True, axis="y", alpha=0.3)

fig7_path = FIG_DIR / f"{NOTEBOOK_NUM}_unstable_gap_windows.png"
fig.savefig(fig7_path, dpi=180, bbox_inches="tight")
plt.show()

fig7_path

## 15. Figure 8 — local stability map

Windowed gap drift for reconstruction methods across scenarios.

In [ ]:
methods_for_heatmap = ["density_only", "local_gap"]
heat_rows = []

for scenario in [cfg["name"] for cfg in SCENARIOS]:
    for method in methods_for_heatmap:
        sub = window_metrics_df[
            (window_metrics_df["scenario"] == scenario)
            & (window_metrics_df["method"] == method)
        ].sort_values("window_mid")
        for _, row in sub.iterrows():
            heat_rows.append({
                "scenario_method": f"{scenario} | {method}",
                "window_mid": row["window_mid"],
                "gap_drift": row["gap_drift"],
            })

heat_df = pd.DataFrame(heat_rows)
pivot = heat_df.pivot(index="scenario_method", columns="window_mid", values="gap_drift")

fig, ax = plt.subplots(figsize=(14, 6))
im = ax.imshow(pivot.values, aspect="auto", interpolation="nearest")
ax.set_yticks(np.arange(len(pivot.index)))
ax.set_yticklabels(pivot.index, fontsize=7)
ax.set_xticks(np.linspace(0, len(pivot.columns)-1, 8).astype(int))
ax.set_xticklabels([int(pivot.columns[i]) for i in np.linspace(0, len(pivot.columns)-1, 8).astype(int)], rotation=45)
ax.set_title("Windowed gap drift stability map")
ax.set_xlabel("window midpoint")
ax.set_ylabel("scenario | method")
fig.colorbar(im, ax=ax, label="gap drift")

fig8_path = FIG_DIR / f"{NOTEBOOK_NUM}_windowed_gap_drift_stability_map.png"
fig.savefig(fig8_path, dpi=180, bbox_inches="tight")
plt.show()

fig8_path

## 16. Interpretation

Notebook 09 shifts from global reconstruction metrics to local-window stability.

Core result:

> Global reconstruction can look successful while local windows expose density, gap, and stability failures.

This gives a stronger test of reconstruction quality than F1 alone.

In [ ]:
interpretation_lines = [
    f"# {NOTEBOOK_TITLE}",
    "",
    "## Constraint result",
    "",
    "This notebook evaluates reconstruction stability in sliding windows.",
    "",
    "Notebook 07 and 08 showed that global F1 and global density can saturate.",
    "",
    "Notebook 09 tests local density error and local gap drift across windows.",
    "",
    "## Main findings",
    "",
    "Global reconstruction metrics are not sufficient.",
    "",
    "Windowed diagnostics expose local instability that global metrics can hide.",
    "",
    "Density-only and local-gap reconstructions should be compared by local density error, local gap drift, and unstable-window counts.",
    "",
    "## Summary metrics",
    "",
    f"- mean window density error, density-only = {measurement['mean_window_density_error_density_only']:.6f}",
    f"- mean window density error, local-gap = {measurement['mean_window_density_error_local_gap']:.6f}",
    f"- mean window gap drift, density-only = {measurement['mean_window_gap_drift_density_only']:.6f}",
    f"- mean window gap drift, local-gap = {measurement['mean_window_gap_drift_local_gap']:.6f}",
    f"- mean delta gap drift local-minus-density = {measurement['mean_delta_gap_drift_local_minus_density']:.6f}",
    f"- mean delta density error local-minus-density = {measurement['mean_delta_density_error_local_minus_density']:.6f}",
    f"- CGCS windowed stability score = {cgcs['score']:.6f}",
    "",
    "## Interpretation",
    "",
    "Sieve filtering removes invalid composites but can create local holes.",
    "",
    "Density reconstruction repairs global mass but can still hide local failures.",
    "",
    "Local-gap reconstruction should be judged by whether it reduces windowed gap drift and unstable windows.",
    "",
    "## Core statement",
    "",
    "Global reconstruction can look successful while local windows expose density, gap, and stability failures.",
    "",
    "## Caution",
    "",
    "Candidate completions remain hypotheses rather than certified primes.",
]

interpretation = "\n".join(interpretation_lines)

figure_paths = [fig1_path, fig2_path, fig3_path, fig4_path, fig5_path, fig6_path, fig7_path, fig8_path]
figure_titles = [
    "Mean window density error by method",
    "Mean window gap drift by method",
    "Window density error profile",
    "Window gap drift profile",
    "Window gap drift delta",
    "Unstable density windows",
    "Unstable gap windows",
    "Windowed gap drift stability map",
]

figures_md = "\n\n## Figures\n\n"
for i, (fig, title) in enumerate(zip(figure_paths, figure_titles), start=1):
    figures_md += f"### Figure {i} — {title}\n\n"
    figures_md += f"![Figure {i}](../figures/{fig.name})\n\n"

print(interpretation + figures_md)

## 17. Export data, docs, math, and TeX

In [ ]:
summary_df = pd.DataFrame([{
    **params,
    **summary,
    **measurement,
    "cgcs_score": cgcs["score"],
    "cgcs_definition": cgcs["definition"],
}])

summary_path = DATA_DIR / f"{NOTEBOOK_NUM}_summary.csv"
window_metrics_path = DATA_DIR / f"{NOTEBOOK_NUM}_window_metrics.csv"
window_summary_path = DATA_DIR / f"{NOTEBOOK_NUM}_window_summary.csv"
window_deltas_path = DATA_DIR / f"{NOTEBOOK_NUM}_window_deltas.csv"
candidate_completions_path = DATA_DIR / f"{NOTEBOOK_NUM}_candidate_completions.csv"
metadata_path = DATA_DIR / f"{NOTEBOOK_NUM}_metadata.json"

interpretation_path = DOCS_DIR / f"{NOTEBOOK_NUM}_interpretation.md"
design_path = DOCS_DIR / f"{NOTEBOOK_NUM}_design_notes.md"

summary_tex_path = TEX_DIR / f"{NOTEBOOK_NUM}_summary_snippet.tex"
math_tex_path = TEX_DIR / f"{NOTEBOOK_NUM}_math_notes.tex"

summary_df.to_csv(summary_path, index=False)
window_metrics_df.to_csv(window_metrics_path, index=False)
window_summary_df.to_csv(window_summary_path, index=False)
window_deltas_df.to_csv(window_deltas_path, index=False)
candidate_completions_df.to_csv(candidate_completions_path, index=False)

metadata = {
    "params": params,
    "summary": summary,
    "measurement": measurement,
    "cgcs": cgcs,
    "figures": [str(p) for p in figure_paths],
    "data": {
        "summary": str(summary_path),
        "window_metrics": str(window_metrics_path),
        "window_summary": str(window_summary_path),
        "window_deltas": str(window_deltas_path),
        "candidate_completions": str(candidate_completions_path),
    },
    "docs": {
        "interpretation": str(interpretation_path),
        "design_notes": str(design_path),
    },
    "tex": {
        "summary_snippet": str(summary_tex_path),
        "math_notes": str(math_tex_path),
    },
}

metadata_path.write_text(json.dumps(metadata, indent=2), encoding="utf-8")
interpretation_path.write_text(interpretation + figures_md + "\n", encoding="utf-8")

design_lines = [
    f"# Design Notes — {NOTEBOOK_TITLE}",
    "",
    "## Notebook role",
    "",
    "Notebook 09 follows Notebook 08 by testing reconstruction stability across sliding windows.",
    "",
    "## Motivation",
    "",
    "Global F1 and global density can saturate. Windowed diagnostics expose local failures.",
    "",
    "## Methods",
    "",
    "1. raw observation",
    "2. sieve-cleaned observation",
    "3. density-only reconstruction",
    "4. local-gap reconstruction",
    "",
    "## Window diagnostics",
    "",
    "1. window density error",
    "2. window gap drift",
    "3. window precision",
    "4. window recovery",
    "5. unstable density-window count",
    "6. unstable gap-window count",
    "",
    "## Core claim",
    "",
    "Global reconstruction can look successful while local windows expose density, gap, and stability failures.",
    "",
    "## Handoff",
    "",
    "Notebook 10 should test adaptive reconstruction by modifying candidate scores in unstable windows.",
]

design_path.write_text("\n".join(design_lines) + "\n", encoding="utf-8")

summary_tex_lines = [
    rf"\section*{{{NOTEBOOK_TITLE}}}",
    "",
    r"This notebook evaluates reconstruction stability across sliding windows.",
    "",
    r"For a window $W=[a,b]$, local density error is",
    r"\[",
    r"d(W)=\frac{||\hat{P}\cap W|-|P\cap W||}{|P\cap W|}.",
    r"\]",
    "",
    r"Local gap drift is",
    r"\[",
    r"g(W)=\operatorname{mean}_{p_i\in W}",
    r"\frac{|(p_{i+1}-p_i)-\log p_i|}{\log p_i}.",
    r"\]",
    "",
    rf"For $N={N_MAX:,}$ with window width ${WINDOW_WIDTH:,}$ and step ${WINDOW_STEP:,}$:",
    r"\begin{itemize}",
    rf"  \item mean density-only window density error $= {measurement['mean_window_density_error_density_only']:.6f}$",
    rf"  \item mean local-gap window density error $= {measurement['mean_window_density_error_local_gap']:.6f}$",
    rf"  \item mean density-only window gap drift $= {measurement['mean_window_gap_drift_density_only']:.6f}$",
    rf"  \item mean local-gap window gap drift $= {measurement['mean_window_gap_drift_local_gap']:.6f}$",
    rf"  \item CGCS windowed stability score $= {cgcs['score']:.6f}$",
    r"\end{itemize}",
    "",
    r"Global reconstruction can look successful while local windows expose density, gap, and stability failures.",
]

summary_tex_path.write_text("\n".join(summary_tex_lines) + "\n", encoding="utf-8")

math_tex_lines = [
    r"\documentclass{article}",
    r"\usepackage{amsmath}",
    r"\usepackage{amssymb}",
    r"\usepackage[margin=1in]{geometry}",
    "",
    r"\begin{document}",
    "",
    r"\section*{Math Notes: Windowed Reconstruction Stability}",
    "",
    r"\subsection*{Window}",
    r"\[W=[a,b].\]",
    "",
    r"\subsection*{Window density error}",
    r"\[",
    r"d(W)=\frac{||\hat{P}\cap W|-|P\cap W||}{|P\cap W|}.",
    r"\]",
    "",
    r"\subsection*{Window precision}",
    r"\[",
    r"precision(W)=\frac{|\hat{P}\cap P\cap W|}{|\hat{P}\cap W|}.",
    r"\]",
    "",
    r"\subsection*{Window recovery}",
    r"\[",
    r"recovery(W)=\frac{|\hat{P}\cap P\cap W|}{|P\cap W|}.",
    r"\]",
    "",
    r"\subsection*{Window gap drift}",
    r"\[",
    r"g(W)=\operatorname{mean}_{p_i\in W}",
    r"\frac{|(p_{i+1}-p_i)-\log p_i|}{\log p_i}.",
    r"\]",
    "",
    r"\subsection*{Windowed stability score}",
    r"\[",
    r"CGCS_{\mathrm{window}}=",
    r"\frac{1}{1+\overline{d(W)}+\overline{g(W)}}.",
    r"\]",
    "",
    r"\end{document}",
]

math_tex_path.write_text("\n".join(math_tex_lines) + "\n", encoding="utf-8")

summary_path, window_metrics_path, window_summary_path, window_deltas_path, candidate_completions_path, metadata_path, interpretation_path, design_path, summary_tex_path, math_tex_path

## 18. Export zip

Pi-stage-lab style root export zip, with optional Colab download lines left commented.

In [ ]:
EXPORT_NAME = f"{NOTEBOOK_ID}_export.zip"

with zipfile.ZipFile(EXPORT_NAME, "w", zipfile.ZIP_DEFLATED) as z:
    for folder in [DOCS_DIR, DATA_DIR, FIG_DIR, TEX_DIR]:
        for path in folder.rglob("*"):
            if path.is_file():
                z.write(path, path.as_posix())

print(f"Export ready: {EXPORT_NAME}")
print("Tip: uncomment Colab lines below to download.")

# --- Optional Colab download ---
# Uncomment the lines below when running in Colab
#
# from google.colab import files
# files.download(EXPORT_NAME)

## 19. Next notebook handoff

Next notebook:

```text
10_adaptive_window_reconstruction.ipynb
```

Purpose:

> use unstable windows to adapt reconstruction scores and repair local failures.

In [ ]:
next_step = "Notebook 10: adaptive window reconstruction."
print(next_step)